# NotebookLM Kit — Notebook Inventory

Read-only view of a notebook's contents.

1. **Setup** — authenticate and verify `tsx`
2. **Config** — set your notebook ID
3. **Sources** — list all sources (indexed)
4. **Artifacts** — list all artifacts with their type, creation time, and the indices of the sources used to generate each one (referencing the table above)

In [ ]:
import importlib, pipeline, pipeline._core, pipeline._sources, pipeline._artifacts, pipeline.config
importlib.reload(pipeline.config)
importlib.reload(pipeline._core)
importlib.reload(pipeline._sources)
importlib.reload(pipeline._artifacts)
importlib.reload(pipeline)

from pipeline import load_credentials, check_tsx

creds = load_credentials(mode="auto")
check_tsx()


credentials: using patchright (credentials.json found)
Credentials ready — token: 42 chars, cookies: 2548 chars
tsx: tsx v4.22.3
node v20.17.0


## Config

Find your notebook ID in the NotebookLM URL:  
`https://notebooklm.google.com/notebook/` **`<YOUR_NOTEBOOK_ID>`**

In [ ]:
NOTEBOOK_ID = "0a79eb08-bbdb-48a0-9b23-5a2ca44e368f"  # ← paste your notebook ID here

## Sources

`list_sources(notebook_id, creds)` — prints an indexed table of all sources.  
The index column is what the **Artifacts** table below uses to reference each source.

In [ ]:
from pipeline import list_sources

sources = list_sources(NOTEBOOK_ID, creds)


Notebook :  4 Napoleon
+---+----------------------+------------+--------------------------------------+
| # | Title                | Status     | Source ID                            |
+---+----------------------+------------+--------------------------------------+
| 0 | 00. Introduction.txt | READY      | 89fd686f-2cd5-4ef0-b427-50424c8d24b4 |
| 1 | 01. Corsica.txt      | READY      | ce3a771d-70b1-44ee-949c-8e83e4a21d52 |
| 2 | 02. Revolution.txt   | READY      | ebd71561-b799-4de4-bde9-1086cef8ae7e |
| 3 | 03. Desire.txt       | READY      | 5f2ccf0c-095a-499a-889f-9abb1bc92525 |
| 4 | 04. Italy.txt        | READY      | 56a7c070-e2d7-41f0-8467-5fe755e38f42 |
| 5 | 05. Victory.txt      | READY      | 36ac048c-22b5-403d-be4e-8d80f1cc68fc |
| 6 | 06. Peace.txt        | READY      | 3677333b-9223-4605-8920-f0b914ab52a5 |
| 7 | 07. Egypt.txt        | READY      | 8606ebfd-a9fb-443e-9779-cda83dac7c94 |
| 8 | 08. Acre.txt         | READY      | 5352bfee-94aa-4f22-bff3-7f9984c43aa6 |
| 9 

## Artifacts

`list_artifacts(notebook_id, sources, creds)` — every artifact in the notebook with:

- **Title** and **Type** (FLASHCARDS, QUIZ, VIDEO, …)
- **Created** — local time pulled from the artifact's own `createdAt`
- **Sources** — comma-separated **indices** into the sources table above (e.g. `0,2` means rows 0 and 2)

In [ ]:
from pipeline import list_artifacts

artifacts = list_artifacts(NOTEBOOK_ID, sources, creds)


Artifacts in notebook 0a79eb08-bbdb-48a0-9b23-5a2ca44e368f (16 total)
+----+---------------------------------------------+-------------+---------------------+---------+--------------------------------------+
| #  | Title                                       | Type        | Created             | Sources | Artifact ID                          |
+----+---------------------------------------------+-------------+---------------------+---------+--------------------------------------+
|  0 | 9 The 18 Brumaire Coup                      | VIDEO       | 2026-05-26 14:27:56 | 9       | caf95f09-ce4a-4f24-a3b1-b4371dae9ebe |
|  1 | 1800 Campaign & Marengo                     | VIDEO       | 2026-05-26 14:28:24 | 11      | 08e226fd-9439-4905-8c20-1a5c9803d7b8 |
|  2 | Rebuilding France                           | VIDEO       | 2026-05-26 14:28:48 | 12      | ed6f3a30-5260-4b11-a08b-c8d006ad6078 |
|  3 | Napoleon Takes Control                      | VIDEO       | 2026-05-26 14:28:06 | 10      | 22

## Rename Single-Source Artifacts

`rename_single_source_artifacts(artifacts, sources, creds, *, indices=None, dry_run=False)`

For every artifact whose `sourceIds` contains exactly **one** source, rename it to:

```
<source title> YYMMDD HHMM
```

where the timestamp comes from the artifact's own `createdAt` (local time).

- `indices` — optional list of row numbers from the Artifacts table above to limit the operation; `None` means all single-source artifacts.
- `dry_run=True` — preview the planned renames without calling the API.

Multi-source artifacts (e.g. audio overviews built from all sources) are left untouched.

In [ ]:
from pipeline import rename_single_source_artifacts

# Preview only — no changes made
rename_single_source_artifacts(artifacts, sources, creds, dry_run=False)

# Limit to specific rows from the Artifacts table:
# rename_single_source_artifacts(artifacts, sources, creds, indices=[0, 1, 3], dry_run=True)


Renaming 16 single-source artifact(s)
+----+---------------------------------------------+----------------------------------+----------+
| #  | Old title                                   | New title                        | Status   |
+----+---------------------------------------------+----------------------------------+----------+
|  0 | 9 The 18 Brumaire Coup                      | 09. Brumaire [260526 142756]     | ok       |
|  1 | 1800 Campaign & Marengo                     | 11. Marengo [260526 142824]      | ok       |
|  2 | Rebuilding France                           | 12. Lawgiver [260526 142848]     | ok       |
|  3 | Napoleon Takes Control                      | 10. Consul [260526 142806]       | ok       |
|  4 | 8 Napoleon at Acre                          | 08. Acre [260525 124020]         | ok       |
|  5 | 7 Napoleon's 1798 Campaign                  | 07. Egypt [260525 124006]        | ok       |
|  6 | 6 Napoleon's Rise to Power                  | 06. Peace [260525

[{'index': 0,
  'artifactId': 'caf95f09-ce4a-4f24-a3b1-b4371dae9ebe',
  'oldTitle': '9 The 18 Brumaire Coup',
  'newTitle': '09. Brumaire [260526 142756]',
  'status': 'ok',
  'error': None},
 {'index': 1,
  'artifactId': '08e226fd-9439-4905-8c20-1a5c9803d7b8',
  'oldTitle': '1800 Campaign & Marengo',
  'newTitle': '11. Marengo [260526 142824]',
  'status': 'ok',
  'error': None},
 {'index': 2,
  'artifactId': 'ed6f3a30-5260-4b11-a08b-c8d006ad6078',
  'oldTitle': 'Rebuilding France',
  'newTitle': '12. Lawgiver [260526 142848]',
  'status': 'ok',
  'error': None},
 {'index': 3,
  'artifactId': '229a5acc-7d9f-44a9-8fab-ce78bd769eca',
  'oldTitle': 'Napoleon Takes Control',
  'newTitle': '10. Consul [260526 142806]',
  'status': 'ok',
  'error': None},
 {'index': 4,
  'artifactId': 'bd723518-e4cc-41e9-a25f-b261ff33a1f5',
  'oldTitle': '8 Napoleon at Acre',
  'newTitle': '08. Acre [260525 124020]',
  'status': 'ok',
  'error': None},
 {'index': 5,
  'artifactId': '86320f79-c5d4-472e-84c5-

## Download Artifacts by Type

download_artifacts_by_type(artifacts, artifact_type, notebook_id, creds, *, indices=None, output_dir=None)

Downloads every artifact of the given type from the rtifacts list (or a subset via indices).
Files land in outputs/<artifact_type_lower>/<Notebook Name>/<yyyymmdd_hhmmss>_<title><ext>.
Already-downloaded files are skipped automatically.

Valid types: FLASHCARDS, QUIZ, AUDIO, INFOGRAPHIC, VIDEO, SLIDE_DECK.

In [ ]:
from pipeline import download_artifacts_by_type

download_artifacts_by_type(artifacts, "Audio", NOTEBOOK_ID, creds)

# Limit to specific rows from the Artifacts table:
# download_artifacts_by_type(artifacts, "VIDEO", NOTEBOOK_ID, creds, indices=[0, 2, 5])
# download_artifacts_by_type(artifacts[2:6], "AUDIO", NOTEBOOK_ID, creds)


Downloaded 2 / 2 artifact(s)  →  d:\Core\_Code D\notebooklm-kit\outputs\audio\Mastering Modern Data Orchestration with Dagster
+---+----------------------------------------------+----------------------------------------------+------------+
| # | Source                                       | File                                         | Status     |
+---+----------------------------------------------+----------------------------------------------+------------+
| 0 | Replacing Airflow Tasks With Dagster Assets  | Replacing Airflow Tasks With Dagster Asse... | downloaded |
| 1 | Dagster Orchestrates Assets Instead Of Tasks | Dagster Orchestrates Assets Instead Of Ta... | downloaded |
+---+----------------------------------------------+----------------------------------------------+------------+


[{'sourceTitle': 'Replacing Airflow Tasks With Dagster Assets',
  'notebookTitle': 'Mastering Modern Data Orchestration with Dagster',
  'createdAt': '2026-02-26T04:44:25.868Z',
  'filePath': 'd:\\Core\\_Code D\\notebooklm-kit\\outputs\\audio\\Mastering Modern Data Orchestration with Dagster\\Replacing Airflow Tasks With Dagster Assets.mp3',
  'status': 'downloaded'},
 {'sourceTitle': 'Dagster Orchestrates Assets Instead Of Tasks',
  'notebookTitle': 'Mastering Modern Data Orchestration with Dagster',
  'createdAt': '2026-02-26T04:44:17.779Z',
  'filePath': 'd:\\Core\\_Code D\\notebooklm-kit\\outputs\\audio\\Mastering Modern Data Orchestration with Dagster\\Dagster Orchestrates Assets Instead Of Tasks.mp3',
  'status': 'downloaded'}]